# Solutions · 09 · Nonlinear regression: fitting the Antoine equation

Worked solutions to the exercises of [notebook 09](../notebooks/09_nonlinear_regression.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere engmath is missing): install it from GitHub
    import engmath
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/engmath"], check=True)
import numpy as np
import matplotlib.pyplot as plt
import engmath
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")
from engmath import fitting
A_t, B_t, C_t = 8.20417, 1642.89, 230.300
T = np.linspace(20, 90, 12)
p = 10**(A_t - B_t/(C_t + T)) * (1 + 0.01*np.random.default_rng(5).standard_normal(T.size))
def antoine_log(T, A, B, C):
    return A - B/(C + T)
fit_log = fitting.levenberg_marquardt(antoine_log, T, np.log10(p), p0=[8.0, 1500.0, 220.0])

## Exercise 1

Fit $p$ directly (not $\log p$) with equal weights. Which data points dominate the fit, and how do
   the parameters change?

In [2]:
fit_p = fitting.levenberg_marquardt(lambda T, A, B, C: 10**(A - B/(C + T)), T, p, p0=[8.0, 1500.0, 220.0])
for name, f in (("fit to log10 p", fit_log), ("fit to p      ", fit_p)):
    rel = 100 * (10**antoine_log(T, *f.coef) / p - 1)
    print(f"{name}: A, B, C = {np.round(f.coef, 3)};  relative residuals at 20 °C {rel[0]:+.2f} %, at 90 °C {rel[-1]:+.2f} %")

fit to log10 p: A, B, C = [   7.929 1487.35   216.327];  relative residuals at 20 °C -0.35 %, at 90 °C +0.95 %
fit to p      : A, B, C = [   7.297 1153.028  182.841];  relative residuals at 20 °C -5.39 %, at 90 °C +0.40 %


With equal weights on $p$, the high-temperature points - whose pressures are twenty times larger - dominate
the sum of squares, and the fit sacrifices the low-temperature end. Because the measurement errors are
*relative* (1 % of $p$), fitting $\log p$ weights all points equally in the right sense: its residuals
stay within about 1 % everywhere, while the fit to $p$ is off by more than 5 % at 20 °C. The two fits
also give very different parameters - with strongly correlated parameters, how the data are weighted
changes the estimates a lot.

## Exercise 2

Fix $C$ at 230.3 and fit only $A$ and $B$ - a linear problem. Compare the standard errors with the
   three-parameter fit.

In [3]:
fitC = fitting.linear_lstsq(np.column_stack([np.ones(T.size), -1/(230.3 + T)]), np.log10(p))
print(f"C fixed at 230.3: A = {fitC.coef[0]:.4f} ± {fitC.se[0]:.4f},  B = {fitC.coef[1]:.2f} ± {fitC.se[1]:.2f}")
print(f"all three free:   A = {fit_log.coef[0]:.4f} ± {fit_log.se[0]:.4f},  B = {fit_log.coef[1]:.2f} ± {fit_log.se[1]:.2f}")

C fixed at 230.3: A = 8.2176 ± 0.0153,  B = 1646.75 ± 4.33
all three free:   A = 7.9291 ± 0.1928,  B = 1487.35 ± 103.59


Fixing $C$ removes the parameter that $A$ and $B$ were trading off against, and their standard errors shrink
by more than an order of magnitude. This is why tabulated Antoine constants often fix $C$, or quote all
three with the understanding that they belong together.

## Exercise 3

Fit the cooling data of notebook 06 by integrating the model instead of differentiating the data:
   fit $\Delta T(t) = [\Delta T_0^{-1/4} + Ct/4]^{-4}$ (generalised to exponent $n$) to the *noisy*
   temperatures. Compare the precision of $n$ with the differential method.

In [4]:
T_inf, dT0, C, n_true = 20.0, 480.0, 5.0e-4, 1.25
t = np.r_[np.arange(0, 60, 5), np.arange(60, 300, 20), np.arange(300, 1801, 100)].astype(float)
T_noisy = T_inf + (dT0**-0.25 + C*t/4)**-4 + np.random.default_rng(3).normal(0, 0.5, t.size)
def cooling(t, dT_0, C_, n):          # solution of dT/dt = -C (T - T_inf)^n for n != 1
    return T_inf + (dT_0**(1 - n) + (n - 1)*C_*t)**(1/(1 - n))
fit = fitting.levenberg_marquardt(cooling, t, T_noisy, [450.0, 1e-3, 1.1])
print(f"integrated model: n = {fit.coef[2]:.4f} ± {fit.se[2]:.4f}   (true 1.25)")
print("differential method on the same data (notebook 06): n = 1.249 ± 0.019")

integrated model: n = 1.2468 ± 0.0029   (true 1.25)
differential method on the same data (notebook 06): n = 1.249 ± 0.019


Fitting the integrated model uses the temperatures directly, without amplifying their noise by
differentiation, and determines $n$ far more precisely than the differential method. The differential
method remains useful for *discovering* the form of a rate law; for estimating its parameters, fit the
integrated model.

## Exercise 4

**Implement it yourself:** run the hand-written loop for 40 iterations from $(7.5, 1300, 180)$ and count the rejected steps - why does it crawl, given the parameter correlations? Then remove the damping ($\lambda = 0$, pure Gauss-Newton) and try both starting points. What does the damping buy you?

In [5]:
def jac(f, x, params, rel=1e-7):
    J = np.empty((len(x), len(params)))
    for j in range(len(params)):
        dp = rel * max(abs(params[j]), 1e-8); sh = np.array(params, float); sh[j] += dp
        J[:, j] = (f(x, *sh) - f(x, *params)) / dp
    return J

def lm_by_hand(start, lam0, iters=40, damping=True):
    y_obs = np.log10(p); params = np.array(start, float); lam = lam0
    rss = np.sum((y_obs - antoine_log(T, *params))**2); rejected = 0
    for _ in range(iters):
        J = jac(antoine_log, T, params); A = J.T @ J
        damp = lam * np.diag(np.diag(A)) if damping else 0
        try:
            step = np.linalg.solve(A + damp, J.T @ (y_obs - antoine_log(T, *params)))
        except np.linalg.LinAlgError:
            return params, rss, rejected
        new = np.sum((y_obs - antoine_log(T, *(params + step)))**2)
        if damping:
            if new < rss: params, rss, lam = params + step, new, lam/10
            else: lam *= 10; rejected += 1
        else:
            params, rss = params + step, new          # Gauss-Newton accepts every step
    return params, rss, rejected

print(f"optimum RSS (library): {np.sum(fit_log.residuals**2):.6e}")
p_lm, rss_lm, rej = lm_by_hand([7.5, 1300, 180], 1e-3)
print(f"LM from (7.5, 1300, 180), 40 iterations: RSS {rss_lm:.6e}, {rej} rejected steps, A, B, C = {np.round(p_lm, 2)}")
with np.errstate(all="ignore"):
    for start in ([8, 1500, 220], [7.5, 1300, 180], [5, 500, 100], [10, 3000, 300]):
        p_gn, rss_gn, _ = lm_by_hand(start, 0.0, iters=15, damping=False)
        print(f"pure Gauss-Newton from {start}, 15 iterations: RSS {rss_gn:.6e}, A, B, C = {np.round(p_gn, 2)}")

optimum RSS (library): 1.373598e-04
LM from (7.5, 1300, 180), 40 iterations: RSS 1.373598e-04, 18 rejected steps, A, B, C = [   7.93 1487.35  216.33]
pure Gauss-Newton from [8, 1500, 220], 15 iterations: RSS 1.373598e-04, A, B, C = [   7.93 1487.35  216.33]
pure Gauss-Newton from [7.5, 1300, 180], 15 iterations: RSS 1.373598e-04, A, B, C = [   7.93 1487.35  216.33]
pure Gauss-Newton from [5, 500, 100], 15 iterations: RSS 1.373598e-04, A, B, C = [   7.93 1487.35  216.33]
pure Gauss-Newton from [10, 3000, 300], 15 iterations: RSS 1.373598e-04, A, B, C = [   7.93 1487.35  216.33]


With correlations near 0.999 the RSS surface is a long, narrow valley. Levenberg-Marquardt insists that
every step lowers the RSS: steps that overshoot the valley floor are rejected, $\lambda$ grows, and progress
becomes a crawl (here it gets there, but needs 40 iterations). Pure Gauss-Newton simply takes every
step - and on this problem it converges quickly from **all** starting points, even very poor ones. The
reason: the Antoine model is linear in $A$ and $B$, and only $C$ enters nonlinearly, so the linearisation
is good almost everywhere, and Gauss-Newton recovers from a temporary overshoot. Damping is insurance:
it costs speed on well-behaved problems like this one, and pays off on strongly nonlinear models or
starts far from the optimum, where undamped steps can diverge.